In [5]:
import numpy as np
from PIL import Image

# Specify the path to your .bin file
file_path = 'y_xsim_int32.bin'

# Read the binary file as int32
data = np.fromfile(file_path, dtype=np.int32)

print(f"Number of int32 elements: {data.size}")
expected_size = 1 * 1 * 50 * 63
print(f"Expected number of elements: {expected_size}")
# Reshape to the expected shape
data = data.reshape((1, 1, 50, 63))

# Optional: print or inspect
print(data.shape)

image_2d = data[0, 0].astype(np.uint8)  # now image_2d.shape == (50, 63)
print(image_2d.shape)

img = Image.fromarray(image_2d, mode='L')  # 'L' = 8-bit pixels, black and white
img.save('output_xsim.png')


Number of int32 elements: 3150
Expected number of elements: 3150
(1, 1, 50, 63)
(50, 63)


In [4]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import torch.nn.functional as F

torch_y = torch.from_numpy(data).to(torch.int32)
output = T.ToPILImage()(torch_y.squeeze(0).to(torch.uint8))
output.save("output_downsampled_q.jpg")

In [10]:
import os
import numpy as np
from numpy.testing import assert_allclose 
from PIL import Image

current_dir = os.getcwd()

output_ref = np.load(f"{current_dir}/y_int32.npy")
output_tensor = np.fromfile(f"{current_dir}/y_xsim_int32.bin", dtype=np.int32).reshape(output_ref.shape)


print(f"Output shape: {output_tensor.shape}, MIN: {output_tensor.min()}, MAX:{output_tensor.max()}\n" )
assert_allclose(output_ref, output_tensor,rtol=1e-6,atol=1e-6)
print("\n****************")
print("* Test passed! *")
print("****************\n")
output_tensor = np.squeeze(output_tensor)  # Remove batch dim → [C, H, W]
img = Image.fromarray(output_tensor.astype(np.uint8), mode='L')  # 'L' = 8-bit pixels, black and white
img.save('output_xsim.png')


Output shape: (1, 1, 50, 63), MIN: 5, MAX:218


****************
* Test passed! *
****************



In [3]:
import os
import numpy as np
from numpy.testing import assert_allclose 
from PIL import Image

image_fname="image_00002.jpg"
path_suffix =""
#path_suffix='conv2d/test'

output_dir = f"{os.getcwd()}/{path_suffix}"
image_path=f"{output_dir}/{image_fname}"


# === LOAD AND CONVERT IMAGE TO RGB ===
img = Image.open(image_path).convert('RGB') 
img_np = np.array(img)  # Shape: (H, W, 3), dtype=uint8

# === PACK RGB TO INT32 ===
# Format: 0x00RRGGBB (most significant byte can be 0)
r = img_np[:, :, 0].astype(np.uint32)
g = img_np[:, :, 1].astype(np.uint32)
b = img_np[:, :, 2].astype(np.uint32)
rgb_packed = (r << 16) | (g << 8) | b  # Shape: (H, W)
print(rgb_packed.shape)

rgb_flat = rgb_packed.flatten().astype(np.uint32)

# === SAVE TO BINARY FILE ===
rgb_flat.tofile(f"{output_dir}/{image_fname}_{rgb_packed.shape[0]}x{rgb_packed.shape[1]}_RGB.bin")

(100, 125)
